In [ ]:
pip install pandas requests biopython openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 44.0 MB/s eta 0:00:00


In [ ]:
!pip install pandas numpy biopython requests openpyxl tqdm

In [ ]:
#!/usr/bin/env python3
"""
Notebook-Friendly PPB-Affinity Sequence Extractor
Keeps the entire original dataset and adds sequence columns
"""

import pandas as pd
import numpy as np
import os
import time
import requests
from Bio import PDB
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import warnings
from pathlib import Path
from tqdm.auto import tqdm
import tempfile

# Suppress ALL PDB warnings (version-independent)
warnings.filterwarnings("ignore")

def create_cache_dir(cache_dir="pdb_cache"):
    """Create cache directory if it doesn't exist."""
    Path(cache_dir).mkdir(exist_ok=True)
    return cache_dir

def download_pdb(pdb_id, cache_dir="pdb_cache"):
    """Download PDB file from RCSB PDB."""
    pdb_id = pdb_id.lower().strip()
    cache_path = os.path.join(cache_dir, f"{pdb_id}.pdb")

    # Check if already cached
    if os.path.exists(cache_path):
        return cache_path

    # Download from RCSB PDB
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        # Save to cache
        with open(cache_path, 'w') as f:
            f.write(response.text)

        return cache_path

    except Exception as e:
        print(f"Failed to download PDB {pdb_id}: {e}")
        return None

def parse_chains(chain_string):
    """Parse chain IDs from string."""
    if pd.isna(chain_string) or not chain_string:
        return []

    # Handle various formats: "A, B", "A,B", "AB", etc.
    chain_str = str(chain_string).strip()

    # Split by common separators
    if ',' in chain_str:
        chains = [c.strip() for c in chain_str.split(',')]
    elif ';' in chain_str:
        chains = [c.strip() for c in chain_str.split(';')]
    elif ' ' in chain_str:
        chains = [c.strip() for c in chain_str.split()]
    else:
        # Assume individual characters are chain IDs
        chains = list(chain_str.replace(' ', ''))

    # Remove empty strings and return
    return [c for c in chains if c]

def extract_sequence_from_chain(structure, chain_id):
    """Extract amino acid sequence from a chain."""
    try:
        for model in structure:
            if chain_id in model:
                chain = model[chain_id]
                residues = []

                for residue in chain:
                    # Only process amino acid residues (not water, ligands, etc.)
                    if residue.get_id()[0] == ' ':  # Standard amino acid
                        try:
                            aa = seq1(residue.get_resname())
                            residues.append(aa)
                        except:
                            residues.append('X')  # Unknown amino acid

                return ''.join(residues)
        return None
    except Exception as e:
        return None

def extract_sequences_for_entry(row, cache_dir="pdb_cache"):
    """Extract sequences for a single dataset entry."""
    try:
        pdb_id = str(row['PDB']).strip()
        ligand_chains = parse_chains(row['Ligand Chains'])
        receptor_chains = parse_chains(row['Receptor Chains'])

        # Check if we have valid chain information
        if not ligand_chains or not receptor_chains:
            return "", "", "Missing chain information"

        # Download PDB file
        pdb_path = download_pdb(pdb_id, cache_dir)
        if not pdb_path:
            return "", "", "PDB download failed"

        # Parse structure
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb_id, pdb_path)

        # Extract ligand sequences
        ligand_seqs = []
        for chain_id in ligand_chains:
            seq = extract_sequence_from_chain(structure, chain_id)
            if seq:
                ligand_seqs.append(seq)

        # Extract receptor sequences
        receptor_seqs = []
        for chain_id in receptor_chains:
            seq = extract_sequence_from_chain(structure, chain_id)
            if seq:
                receptor_seqs.append(seq)

        # Join multiple chains with separator
        ligand_seq = '|'.join(ligand_seqs) if ligand_seqs else ""
        receptor_seq = '|'.join(receptor_seqs) if receptor_seqs else ""

        # Determine status
        if ligand_seq and receptor_seq:
            status = "Success"
        elif ligand_seq or receptor_seq:
            status = "Partial extraction"
        else:
            status = "Sequence extraction failed"

        return ligand_seq, receptor_seq, status

    except Exception as e:
        return "", "", f"Error: {str(e)}"

def convert_kd_to_pkd(kd_value):
    """Convert KD value to pKd."""
    try:
        kd_float = float(kd_value)
        if kd_float > 0:
            return -np.log10(kd_float)
        else:
            return np.nan
    except:
        return np.nan

def extract_sequences_for_dataset(input_file, cache_dir="pdb_cache", sample_size=None):
    """
    Extract sequences for the entire dataset while preserving all original data.

    Parameters:
    -----------
    input_file : str
        Path to the input Excel/CSV file
    cache_dir : str
        Directory to cache PDB files
    sample_size : int or None
        Number of entries to process (None for all)

    Returns:
    --------
    pandas.DataFrame
        Original dataset with added sequence columns
    """

    print("🔬 PPB-Affinity Sequence Extractor")
    print("=" * 50)

    # Create cache directory
    cache_dir = create_cache_dir(cache_dir)
    print(f"📁 Cache directory: {cache_dir}")

    # Load dataset
    print(f"📖 Loading dataset from {input_file}...")
    try:
        if input_file.endswith('.xlsx'):
            df = pd.read_excel(input_file)
        else:
            df = pd.read_csv(input_file)
    except Exception as e:
        print(f"❌ Error loading dataset: {e}")
        return None

    print(f"📊 Loaded {len(df)} entries")
    print(f"📋 Columns: {list(df.columns)}")

    # Sample if requested
    original_length = len(df)
    if sample_size and sample_size < len(df):
        df = df.head(sample_size).copy()
        print(f"🎯 Processing sample of {len(df)} entries (out of {original_length})")

    # Add new columns for sequences and metadata
    df['Target'] = ""  # Ligand sequence
    df['proteina'] = ""  # Receptor sequence
    df['Y'] = np.nan  # pKd value
    df['Sequence_Status'] = ""  # Extraction status
    df['Target_Length'] = 0  # Ligand sequence length
    df['proteina_Length'] = 0  # Receptor sequence length

    # Convert existing KD values to pKd
    if 'KD(M)' in df.columns:
        print("🔢 Converting KD to pKd values...")
        df['Y'] = df['KD(M)'].apply(convert_kd_to_pkd)

    # Process each entry
    print("🧬 Extracting sequences...")
    successful = 0
    failed = 0

    # Use tqdm for progress bar
    for idx in tqdm(range(len(df)), desc="Extracting sequences"):
        row = df.iloc[idx]

        try:
            # Extract sequences
            ligand_seq, receptor_seq, status = extract_sequences_for_entry(row, cache_dir)

            # Update dataframe
            df.at[df.index[idx], 'Target'] = ligand_seq
            df.at[df.index[idx], 'proteina'] = receptor_seq
            df.at[df.index[idx], 'Sequence_Status'] = status
            df.at[df.index[idx], 'Target_Length'] = len(ligand_seq) if ligand_seq else 0
            df.at[df.index[idx], 'proteina_Length'] = len(receptor_seq) if receptor_seq else 0

            if status == "Success":
                successful += 1
            else:
                failed += 1

            # Small delay to be respectful to the server
            if idx % 10 == 9:  # Every 10 requests
                time.sleep(0.5)

        except Exception as e:
            df.at[df.index[idx], 'Sequence_Status'] = f"Error: {str(e)}"
            failed += 1

    # Print summary statistics
    print("\n📈 Extraction Summary")
    print("=" * 30)
    print(f"✅ Successful extractions: {successful}")
    print(f"❌ Failed extractions: {failed}")
    print(f"📊 Success rate: {successful/(successful+failed)*100:.1f}%")

    # Show status distribution
    print(f"\n📋 Status Distribution:")
    status_counts = df['Sequence_Status'].value_counts()
    for status, count in status_counts.items():
        print(f"   {status}: {count}")

    # Show sequence length statistics for successful extractions
    successful_df = df[df['Sequence_Status'] == 'Success']
    if len(successful_df) > 0:
        print(f"\n🧬 Sequence Statistics (Successful Extractions):")
        print(f"   Target (Ligand) length: {successful_df['Target_Length'].mean():.1f} ± {successful_df['Target_Length'].std():.1f} amino acids")
        print(f"   proteina (Receptor) length: {successful_df['proteina_Length'].mean():.1f} ± {successful_df['proteina_Length'].std():.1f} amino acids")

        if 'Y' in df.columns and not df['Y'].isna().all():
            valid_y = successful_df['Y'].dropna()
            if len(valid_y) > 0:
                print(f"   pKd range: {valid_y.min():.2f} to {valid_y.max():.2f}")
                print(f"   pKd mean: {valid_y.mean():.2f} ± {valid_y.std():.2f}")

    return df

def save_results(df, base_filename="ppb_affinity_with_sequences"):
    """Save the results in multiple formats."""

    print(f"\n💾 Saving results...")

    # Save complete dataset
    all_file = f"{base_filename}.csv"
    df.to_csv(all_file, index=False)
    print(f"   📁 All entries: {all_file}")

    # Save successful extractions only
    successful_df = df[df['Sequence_Status'] == 'Success'].copy()
    if len(successful_df) > 0:
        success_file = f"{base_filename}_successful.csv"
        successful_df.to_csv(success_file, index=False)
        print(f"   ✅ Successful only: {success_file}")

        # Save BALM format (Target, proteina, Y)
        balm_columns = ['Target', 'proteina', 'Y']
        if all(col in successful_df.columns for col in balm_columns):
            # Filter for entries with all required data
            balm_ready = successful_df.dropna(subset=['Y'])
            balm_ready = balm_ready[(balm_ready['Target'] != "") & (balm_ready['proteina'] != "")]

            if len(balm_ready) > 0:
                balm_file = f"{base_filename}_balm_format.csv"
                balm_ready.to_csv(balm_file, index=False)
                print(f"   🤖 BALM ready: {balm_file} ({len(balm_ready)} entries)")

                # Save minimal version with just the three columns
                minimal_file = "Data.csv"  # This matches your BALM code expectations
                balm_ready[balm_columns].to_csv(minimal_file, index=False)
                print(f"   📊 Minimal (Data.csv): {minimal_file}")

    # Save summary report
    summary_file = f"{base_filename}_summary.txt"
    with open(summary_file, 'w') as f:
        f.write("PPB-Affinity Sequence Extraction Summary\n")
        f.write("=" * 50 + "\n\n")

        f.write(f"Total entries processed: {len(df)}\n")
        f.write(f"Successful extractions: {(df['Sequence_Status'] == 'Success').sum()}\n")
        f.write(f"Failed extractions: {(df['Sequence_Status'] != 'Success').sum()}\n\n")

        f.write("Status Distribution:\n")
        status_counts = df['Sequence_Status'].value_counts()
        for status, count in status_counts.items():
            f.write(f"  {status}: {count}\n")

        # Add sequence statistics
        successful_df = df[df['Sequence_Status'] == 'Success']
        if len(successful_df) > 0:
            f.write(f"\nSequence Statistics:\n")
            f.write(f"  Average Target length: {successful_df['Target_Length'].mean():.1f} amino acids\n")
            f.write(f"  Average proteina length: {successful_df['proteina_Length'].mean():.1f} amino acids\n")

    print(f"   📄 Summary: {summary_file}")

def show_sample_results(df, n=5):
    """Show a sample of the results."""
    successful_df = df[df['Sequence_Status'] == 'Success']

    if len(successful_df) == 0:
        print("❌ No successful extractions to display")
        return

    print(f"\n🔍 Sample Results (showing {min(n, len(successful_df))} successful entries):")
    print("=" * 80)

    for i, (idx, row) in enumerate(successful_df.head(n).iterrows()):
        print(f"\nEntry {i+1}:")
        print(f"   PDB: {row['PDB']}")
        print(f"   Ligand Chains: {row.get('Ligand Chains', 'N/A')}")
        print(f"   Receptor Chains: {row.get('Receptor Chains', 'N/A')}")
        print(f"   Target Sequence: {row['Target'][:50]}{'...' if len(row['Target']) > 50 else ''}")
        print(f"   proteina Sequence: {row['proteina'][:50]}{'...' if len(row['proteina']) > 50 else ''}")
        print(f"   Y (pKd): {row['Y']:.2f}" if not pd.isna(row['Y']) else "   Y (pKd): N/A")
        print(f"   Lengths: Target={row['Target_Length']}, proteina={row['proteina_Length']}")

# Example usage functions
def quick_test(input_file, n_samples=10):
    """Quick test with a small sample."""
    print(f"🧪 Quick test with {n_samples} samples...")
    df = extract_sequences_for_dataset(input_file, sample_size=n_samples)
    if df is not None:
        show_sample_results(df, n=3)
        return df
    return None

def process_full_dataset(input_file):
    """Process the complete dataset."""
    print("🔬 Processing full dataset...")
    df = extract_sequences_for_dataset(input_file)
    if df is not None:
        save_results(df)
        show_sample_results(df)
        return df
    return None

# Main execution
if __name__ == "__main__":
    # Configuration - MODIFY THESE
    INPUT_FILE = "/content/PPB-Affinity (5).xlsx"  # Your input file
    SAMPLE_SIZE = None  # Set to number for testing, None for all

    # Run extraction
    print("Starting sequence extraction...")
    df_result = extract_sequences_for_dataset(INPUT_FILE, sample_size=SAMPLE_SIZE)

    if df_result is not None:
        # Save results
        save_results(df_result)

        # Show sample
        show_sample_results(df_result)

        print("\n🎉 Extraction completed successfully!")
    else:
        print("❌ Extraction failed!")

Starting sequence extraction...
🔬 PPB-Affinity Sequence Extractor
📁 Cache directory: pdb_cache
📖 Loading dataset from /content/PPB-Affinity (5).xlsx...
📊 Loaded 12062 entries
📋 Columns: ['Unnamed: 0', 'Source Data Set', 'Complex ID', 'PDB', 'Mutations', 'Ligand Chains', 'Receptor Chains', 'Ligand Name', 'Receptor Name', 'KD(M)', 'Affinity Method', 'Structure Method', 'Temperature(K)', 'Resolution(Å)', 'PDB PubMed ID', 'PDB Release Date', 'Affinity PubMed ID', 'Affinity Release Date', 'Subgroup']
🔢 Converting KD to pKd values...
🧬 Extracting sequences...


Extracting sequences:   0%|          | 0/12062 [00:00<?, ?it/s]

Failed to download PDB 4fqr: 404 Client Error: Not Found for url: https://files.rcsb.org/download/4fqr.pdb
Failed to download PDB 4fqr: 404 Client Error: Not Found for url: https://files.rcsb.org/download/4fqr.pdb
Failed to download PDB 4fqr: 404 Client Error: Not Found for url: https://files.rcsb.org/download/4fqr.pdb
Failed to download PDB 4fqr: 404 Client Error: Not Found for url: https://files.rcsb.org/download/4fqr.pdb
Failed to download PDB 4fqr: 404 Client Error: Not Found for url: https://files.rcsb.org/download/4fqr.pdb
Failed to download PDB 4fqr: 404 Client Error: Not Found for url: https://files.rcsb.org/download/4fqr.pdb
Failed to download PDB 4fqr: 404 Client Error: Not Found for url: https://files.rcsb.org/download/4fqr.pdb
Failed to download PDB 4fqr: 404 Client Error: Not Found for url: https://files.rcsb.org/download/4fqr.pdb
Failed to download PDB 4fqr: 404 Client Error: Not Found for url: https://files.rcsb.org/download/4fqr.pdb
Failed to download PDB 4fqr: 404 Clie